<a href="https://colab.research.google.com/github/oooinr4018-web/-1/blob/main/ESAA_0928_%EC%88%98%EC%83%81%EC%9E%91%EB%A6%AC%EB%B7%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 주제

전기차 구매 여부 예측

- 연령, 소득, 통근 거리, 차량 보유 수, 충전소 접근성, 환경 관심도 등의 데이터를 활용하여 Will_Buy_EV 예측

- TabPFNClassifier를 중심으로 Target Encoding과 다양한 파생변수ㅡㄹ 결합하여 분류

- 최종적으로 test 데이터에 대해 전기차를 구매할 확률을 예측

# 데이터

1. Competition 데이터

- train.csv: 668,665행 X 15열

- test.csv: 286,571행 X 14열

- Target: Will_Buy_EV

- Target 분포

No: 약 82.5%

Yes: 약 17.5%

- 주요 변수

Age, Annual_Income_USD, Daily_Commute_km, Number_of_Cars_Owned, Charging_Stations_Near_Home, Charging_Stations_Near_Work, Environmental_concern_level, Gender, City_Type, Current_Car_Type, Home_Charging_Possible, Subsidy_Available, Range_Anxiety_Level

- Target인 Will_Buy_EV는 "No" -> 0, "Yes" -> 1로 변환하여 학습에 사용했다.

# 코드 흐름

1. 데이터 로드 및 Target 설정

- train / test 데이터 로드

- id와 Target을 제외하여 입력 데이터 X 생성

- Will_Buy_EV를 0/1 이진 Target으로 변환

In [ ]:
X=train.drop(columns=['Will_Buy_EV', 'id'])
y=train['Will_Buy_EV'].map({"No":0, "Yes":1})

2. Target Encoding

범주형 변수들을 단순한 문자열 값으로 사용하는 대신, 각 범주와 전기차 구매 여부의 관계를 수치화했다.

In [ ]:
def target_encoding(df, target, col, smoothing):
  global_mean=df[target].mean()
  stats=df.groupby(col)[target].agg(["mean","count"])

  stats["encoding"]=(
      stats["count"]*stats["mean"]
      + smoothing*golbal_mean
  ) / (stats["count"]+smooting)

  return stats["encoding"], global_mean

- 각 범주의 Target 평균을 이용해 Encoding

- 데이터 수가 적은 범주가 지나치게 극단적인 값을 가지지 않도록 global mean과 smoothing

- Gender, City_Type, Current_Car_Type 등의 범주형 변수를 수치화

3. 다양한 파생변수 생성

단순 원본 변수만 사용하지 않고 변수 사이의 관계를 표현하기 위해 파생변수를 대량 생성했다.


Interaction Feature

In [ ]:
poly=PolynomialFeatures(
    interaction_only=True,
    include_bias=False
)

X_num=pd.DataFrame(
    poly.fit_transform(X[num_columns])
)

- 두 변수의 곱을 이용한 상호작용 변수 생성

ex)

- Age x Annual_Income_USD

- Age x Daily_Commute_km

- Income x Environmental_Concern_Level


Digit Feature

In [ ]:
df[f"{col}_mod10"] = df[col] % 10
df[f"{col}_mod100"] = df[col] % 100

- 숫자의 마지막 자리나 마지막 두 자리 정보를 새로운 변수로 생성
- Age, Annual_Income_USD, Daily_Commute_km 등에 적용

sqrt / log 변환

In [ ]:
X_num[f'sqrt_{col}'] = np.sqrt(X_num[col])
X_num[f'log_{col}'] = np.log1p(X_num[col])

- 수치형 변수에 제곱근 및 로그 변환 추가
- 원래 변수와 다른 형태의 비선형 관계를 모델이 활용할 수 있도록 함

4. 상관관계를 이용한 Feature Selection


파생변수를 매우 많이 만들었기 때문에 모든 변수를 사용하지 않고, Target과의 상관관계를 이용해 일부 변수만 선택했다.

In [ ]:
df_corr = df_corr.corr()

features = set(
    df_corr[df_corr.loc[:, 'target'] > 0.45].index
).union(
    df_corr[df_corr.loc[:, 'target'] < -0.04].index
)

- Target과 양의 상관관계가 0.45보다 큰 변수

- 또는 음의 상관관계가 -0.04보다 작은 변수

- 해당 변수만 최종 입력 feature로 사용

-> 많은 파생변수를 생성하면, Target과 관련성이 높은 변수만 선택하는 구조

#모델 학습

5. TabPFNClassifier

이 코드의 핵심 모델은 TabPFN_3

In [ ]:
clf = TabPFNClassifier(
    model_path=MODEL,
    device="cuda",
    n_estimators=4,
    kv_cache_precision="int8",
    random_state=42,
    fit_mode="fit_with_cache"
)

- Prior-Labs에서 개발한 TabPFN_3 사용

- Hugging Face에서 사전 학습된 모델 체크포인트 다운로드

- 표 형태(Tabular) 데이터 분류에 특화된 모델

- GPU를 이용하여 학습

# 새롭게 알게 된 내용 / 어려운 내용 / 배울 점

1. 새롭게 알게 된 내용

- TabPFN이라는 표형 데이터 전용 사전학습 모델을 처음 확인했다.

- 단순 원본 변수만 사용하는 것이 아니라 interaction, log, sqrt, modulo 변수 등 다양한 형태의 파생변수를 만들 수 있다는 것을 알게 되었다.

- Target Encoding에서 단순 평균을 사용하는 것보다 smoothing을 적용해 표본 수가 적은 범주의 값을 안정화할 수 있다는 점을 배웠다.

- 많은 파생변수를 만든 후 Target과의 상관관계를 이용해 다시 feature를 선택하는 방법을 확인했다.

# 어려웠던 내용

-PolynomialFeatures를 적용하면서 매우 많은 interaction feature가 생성되어 전체 변수의 구조를 이해하기 어려웠다.

- 원본 변수뿐 아니라 Target Encoding 변수, interaction 변수, sqrt/log 변수까지 연속해서 생성되기 때문에 각 feature가 어떤 의미인지 추적하는 과정이 복잡했다.

- Train에서 사용한 feature engineering을 Test에도 완전히 동일한 방식과 동일한 순서로 적용해야 한다는 점이 중요했다.

# 배울 점

- 모델 자체를 바꾸는 것뿐 아니라 Feature Engineering이 성능에 큰 영향을 줄 수 있다.

- 범주형 변수는 Target Encoding을 통해 Target과의 관계를 직접적인 수치 정보로 표현할 수 있다.

- 많은 파생변수를 무조건 사용하는 것보다 관련성이 높은 feature를 선별하는 과정도 중요하다.

- 대규모 데이터에서는 전체 데이터를 한 번에 처리하기보다 Batch Prediction을 통해 메모리를 관리하는 방법이 유용하다.

- Train과 Test에는 반드시 동일한 전처리와 동일한 Feature Engineering을 적용해야 한다.